In [5]:
from pyspark.sql import functions as F

silver_df = spark.read.format("delta").load(
    "abfss://silver@idestorageaccount.dfs.core.windows.net/nyc_yellow_taxi/"
)
display(silver_df.limit(5))

StatementMeta(idesparkpool, 2, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2aee353b-3eef-4f7d-b8a2-12af621db8c4)

In [6]:
from pyspark.sql import functions as F

# ---------- dim_date ----------
dim_date = (
    silver_df.select(F.to_date("tpep_pickup_datetime").alias("full_date"))
    .union(silver_df.select(F.to_date("tpep_dropoff_datetime").alias("full_date")))
    .distinct()
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.col("day_of_week").isin([1, 7]))
)
dim_date.write.format("delta").mode("overwrite").save(
    "abfss://gold@idestorageaccount.dfs.core.windows.net/dim_date/"
)

# ---------- dim_time ----------
dim_time = spark.range(0, 1440).select(
    (F.col("id")).alias("time_key"),
    (F.col("id") / 60).cast("int").alias("hour"),
    (F.col("id") % 60).cast("int").alias("minute")
).withColumn("am_pm", F.when(F.col("hour") < 12, "AM").otherwise("PM"))
dim_time.write.format("delta").mode("overwrite").save(
    "abfss://gold@idestorageaccount.dfs.core.windows.net/dim_time/"
)

# ---------- dim_location ----------
dim_location = (
    silver_df.select(F.col("PULocationID").alias("location_key"))
    .union(silver_df.select(F.col("DOLocationID").alias("location_key")))
    .distinct()
)
dim_location.write.format("delta").mode("overwrite").save(
    "abfss://gold@idestorageaccount.dfs.core.windows.net/dim_location/"
)

# ---------- dim_vendor ----------
dim_vendor = spark.createDataFrame([
    (1, "Creative Mobile Technologies"),
    (2, "Curb Mobility"),
    (6, "Myle Technologies"),
    (7, "Helix")
], ["vendor_key", "vendor_name"])
dim_vendor.write.format("delta").mode("overwrite").save(
    "abfss://gold@idestorageaccount.dfs.core.windows.net/dim_vendor/"
)

# ---------- dim_ratecode ----------
dim_ratecode = spark.createDataFrame([
    (1, "Standard rate"), (2, "JFK"), (3, "Newark"),
    (4, "Nassau/Westchester"), (5, "Negotiated fare"),
    (6, "Group ride"), (99, "Unknown")
], ["ratecode_key", "ratecode_desc"])
dim_ratecode.write.format("delta").mode("overwrite").save(
    "abfss://gold@idestorageaccount.dfs.core.windows.net/dim_ratecode/"
)

# ---------- dim_payment_type ----------
dim_payment_type = spark.createDataFrame([
    (1, "Credit card"), (2, "Cash"), (3, "No charge"),
    (4, "Dispute"), (5, "Unknown"), (6, "Voided trip")
], ["payment_type_key", "payment_desc"])
dim_payment_type.write.format("delta").mode("overwrite").save(
    "abfss://gold@idestorageaccount.dfs.core.windows.net/dim_payment_type/"
)

# ---------- fact_trips ----------
fact_trips = (
    silver_df
    .withColumn("trip_id", F.monotonically_increasing_id())
    .withColumn("pickup_date_key", F.date_format("tpep_pickup_datetime", "yyyyMMdd").cast("int"))
    .withColumn("dropoff_date_key", F.date_format("tpep_dropoff_datetime", "yyyyMMdd").cast("int"))
    .withColumn("pickup_time_key", (F.hour("tpep_pickup_datetime") * 60 + F.minute("tpep_pickup_datetime")))
    .withColumn("dropoff_time_key", (F.hour("tpep_dropoff_datetime") * 60 + F.minute("tpep_dropoff_datetime")))
    .select(
        "trip_id", "pickup_date_key", "dropoff_date_key", "pickup_time_key", "dropoff_time_key",
        F.col("PULocationID").alias("pu_location_key"),
        F.col("DOLocationID").alias("do_location_key"),
        F.col("VendorID").alias("vendor_key"),
        F.col("RatecodeID").alias("ratecode_key"),
        F.col("payment_type").alias("payment_type_key"),
        "passenger_count", "trip_distance", "trip_duration_minutes",
        "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "congestion_surcharge",
        F.col("Airport_fee").alias("airport_fee"), "total_amount"
    )
)
fact_trips.write.format("delta").mode("overwrite").save(
    "abfss://gold@idestorageaccount.dfs.core.windows.net/fact_trips/"
)

fact_trips.count()

StatementMeta(idesparkpool, 2, 7, Finished, Available, Finished, False)

3439925